# IF_DIRECT_DOWN Operation Analysis
Analysis script focused exclusively on IF_DIRECT_DOWN failure detection.
Based on EIBPRecoveryAnalysis_updated8.

## Input Required Information

| Variable | Use |
| --- | --- |
| LOG_DIR_PATH | Location of the log directory. |
| END_TIME | Stop time from stop_time.txt on the failed node. |

In [1]:
from If_Direct_Down_AnalysisHelper import *
LOG_DIR_PATH = "/home/fabric/work/NShenoy/Failure_HW_SW/EIBP_OSPF/EIBP_hw_sw/logs"

END_TIME = 1775846514.699300



## Definitions

- **Convergence time**: time taken from the first node detecting IF_DIRECT_DOWN to the last node hearing about it.
- **Control Overhead**: Size of the failure message overall.
- **Churn rate**: how far the error message propagates.
- **IF_DIRECT_DOWN**: nodes detect interface is directly down (link failure detected immediately).
- **Cutoff Time**: earliest IF_DIRECT_DOWN timestamp.

# Convergence Time

In [2]:

# Run IF_DIRECT_DOWN analysis
calculate_if_direct_down_convergence(LOG_DIR_PATH, END_TIME, time_buffer=1.0)


===== All IF_DIRECT_DOWN Events =====
  A1: 1775846469.780874 (delta: +0.000000s) - EIBP_A1.log

Failure type: IF_DIRECT_DOWN
Failing node (auto-detected): A1
Extracted IF_DIRECT_DOWN timestamp: 1775846469.780874
From file: /home/fabric/work/NShenoy/Failure_HW_SW/EIBP_OSPF/EIBP_hw_sw/logs/EIBP_A1.log
END_TIME (stop time): 1775846514.6993
Analysis window: [1775846469.780874, 1775846514.699300] (44.92 seconds)

===== Post-Failure Neighbor State (after delete processing) =====
A1 at 1775846469.780914 - Lost neighbors: ['2.1.2', '2.3.4']
D1 at 1775846469.826428 - Lost neighbors: ['3.1.2.1', '3.1.3.4', '3.3.4.1', '3.3.5.4']
D2 at 1775846469.804892 - Lost neighbors: ['3.1.2.1', '3.3.4.1']

===== Post-Failure Routing State (within analysis window) =====
A1 at 1775846469.780914 - Lost entries: ['3.1.2.1', '3.3.4.1']
C1 at 1775846469.805542 - Lost entries: ['3.1.2.1', '3.3.4.1']
C2 at 1775846469.805916 - Lost entries: ['3.1.2.1', '3.3.4.1']
C3 at 1775846469.805171 - Lost entries: ['3.1.2.1', '

In [3]:
import re
import os

# Regex patterns
pattern_CURRENT_TIME = re.compile(r'CURRENT_TIME:(\d+\.\d+)')
pattern_IF_DOWN = re.compile(r'IF_DIRECT_DOWN:(\d+\.\d+)')

# Trigger messages
trigger_messages = [
    "Received MESSAGE_TYPE_PUBLISH_IP_DELETE",
    "Received MESSAGE_TYPE_MY_LABELS_DELETE"
]

START_TIME = None
start_file = None

# ---------------------------------------------------
# PASS 1: Find earliest IF_DIRECT_DOWN across all files
# ---------------------------------------------------

for filename in os.listdir(LOG_DIR_PATH):
    if filename.endswith(".log"):
        file_path = os.path.join(LOG_DIR_PATH, filename)

        with open(file_path, 'r') as file:
            for line in file:
                match = pattern_IF_DOWN.search(line.strip())
                if match:
                    ts = float(match.group(1))
                    if ts < END_TIME and (START_TIME is None or ts < START_TIME):
                        START_TIME = ts
                        start_file = filename
                    break  # only take first IF_DIRECT_DOWN per file

if START_TIME is None:
    print("ERROR: IF_DIRECT_DOWN not found in any log file.")
    raise SystemExit

print(f"IF_DIRECT_DOWN found in '{start_file}'")
print(f"START_TIME set to: {START_TIME}")
print(f"\nAnalysis Window:")
print(f"START_TIME = {START_TIME}")
print(f"END_TIME   = {END_TIME}\n")


# ---------------------------------------------------
# PASS 2: Process ALL logs within the time window
# ---------------------------------------------------

total_eth_sizes = 0

for filename in os.listdir(LOG_DIR_PATH):

    if filename.endswith(".log"):

        file_path = os.path.join(LOG_DIR_PATH, filename)

        eth_sizes = []
        waiting_for_eth = False
        current_time = None
        line_number = 0

        with open(file_path, 'r') as file:

            for line in file:
                line_number += 1
                line = line.strip()

                # Update CURRENT_TIME
                match = pattern_CURRENT_TIME.search(line)
                if match:
                    current_time = float(match.group(1))

                if current_time is None:
                    continue

                # Skip before window
                if current_time < START_TIME:
                    continue

                # Stop after window
                if current_time >= END_TIME:
                    break

                # Detect trigger messages
                if any(msg in line for msg in trigger_messages):
                    waiting_for_eth = True
                    continue

                # Capture first ETH_SIZE after trigger
                if waiting_for_eth and "ETH_SIZE" in line:
                    try:
                        eth_size = int(line.split(":")[1].strip())
                        eth_sizes.append(eth_size)
                        print(f"Line {line_number} in '{filename}': ETH_SIZE = {eth_size} bytes")
                        waiting_for_eth = False
                    except (IndexError, ValueError):
                        print(f"Error parsing ETH_SIZE in line {line_number} of '{filename}'")

        sum_eth_sizes = sum(eth_sizes)
        total_eth_sizes += sum_eth_sizes

        print(f"\nTotal ETH_SIZE for file '{filename}': {sum_eth_sizes} bytes")

print(f"\nOverall total ETH_SIZE across all files: {total_eth_sizes} bytes")

IF_DIRECT_DOWN found in 'EIBP_A1.log'
START_TIME set to: 1775846469.780874

Analysis Window:
START_TIME = 1775846469.780874
END_TIME   = 1775846514.6993

Line 407 in 'EIBP_A5.log': ETH_SIZE = 32 bytes
Line 419 in 'EIBP_A5.log': ETH_SIZE = 32 bytes

Total ETH_SIZE for file 'EIBP_A5.log': 64 bytes

Total ETH_SIZE for file 'EIBP_A1.log': 0 bytes
Line 1043 in 'EIBP_D2.log': ETH_SIZE = 32 bytes

Total ETH_SIZE for file 'EIBP_D2.log': 32 bytes
Line 1627 in 'EIBP_C1.log': ETH_SIZE = 32 bytes
Line 1640 in 'EIBP_C1.log': ETH_SIZE = 32 bytes

Total ETH_SIZE for file 'EIBP_C1.log': 64 bytes
Line 974 in 'EIBP_D1.log': ETH_SIZE = 32 bytes
Line 986 in 'EIBP_D1.log': ETH_SIZE = 32 bytes

Total ETH_SIZE for file 'EIBP_D1.log': 64 bytes
Line 396 in 'EIBP_A4.log': ETH_SIZE = 32 bytes
Line 408 in 'EIBP_A4.log': ETH_SIZE = 32 bytes

Total ETH_SIZE for file 'EIBP_A4.log': 64 bytes
Line 905 in 'EIBP_D5.log': ETH_SIZE = 32 bytes
Line 917 in 'EIBP_D5.log': ETH_SIZE = 32 bytes

Total ETH_SIZE for file 'EIBP_D5

# CHURN PERCENTAGE

In [4]:
import re
import os
from collections import Counter

failure_time, _, _ = extract_if_direct_down_time(LOG_DIR_PATH, END_TIME)

# Execute IF_DIRECT_DOWN churn analysis
churn_metrics = calculate_churn(LOG_DIR_PATH, failure_time, end_time=END_TIME, time_buffer=1.0)


===== All IF_DIRECT_DOWN Events =====
  A1: 1775846469.780874 (delta: +0.000000s) - EIBP_A1.log
                              IF_DIRECT_DOWN CHURN ANALYSIS

Failure type: IF_DIRECT_DOWN
Failing node: A1
Using failure_time: 1775846469.780874
Using end_time: 1775846514.6993
Analysis window: [1775846469.780874, 1775846514.699300]
Pre-failure nodes found: 13
Post-failure nodes found: 13

===== Neighbor Table Changes Due to IF_DIRECT_DOWN Failure =====

A1: Lost 2 neighbors: ['2.1.2', '2.3.4']

D1: Lost 4 neighbors: ['3.1.2.1', '3.1.3.4', '3.3.4.1', '3.3.5.4']

D2: Lost 2 neighbors: ['3.1.2.1', '3.3.4.1']

===== Routing Table (print_entries_LL) Changes Due to IF_DIRECT_DOWN Failure =====

A1: Lost 2 labels: ['3.1.2.1', '3.3.4.1']

C1: Lost 2 labels: ['3.1.2.1', '3.3.4.1']

C2: Lost 2 labels: ['3.1.2.1', '3.3.4.1']

C3: Lost 2 labels: ['3.1.2.1', '3.3.4.1']

                              IF_DIRECT_DOWN CHURN SUMMARY

Total nodes in network: 13
Nodes that experienced churn: 6
  - Neighbor ta